### **Cuaderno B — Zero-shot con prompts, ensembles, comparación de checkpoints y búsqueda semántica**

**Examen Parcial Virtual MCC225A — Oscar Benito Toledo Guerrero — Tema: CLIP**

#### Trazabilidad

| Sección | Adaptado de | Qué cambié |
|---|---|---|
| Etiquetas operativas | **C9/C10** (columna `label` del bootstrap) | El bootstrap del curso traía labels curadas; aquí las derivo por palabras clave de las captions de Flickr8k (etiqueta operativa, mismo espíritu) |
| Zero-shot con templates | **C10** §4 (`prompt_config.json` + `04_eval_zeroshot.py`) | Plantillas propias añadidas; comparación prompt único vs ensemble en una sola tabla |
| Prompt ensemble | **C10** §4 / paper §3.1.4 | Implementación explícita del promedio + **re-normalización** de embeddings de clase |
| Comparación de checkpoints | **C10** §5 (`05_compare_checkpoints.py`) | `laion2b_s34b_b79k` vs `openai` con las mismas imágenes y prompts |
| Búsqueda semántica | **C10** §6 (`06_build_faiss_index.py`) | FAISS si está instalado; fallback exacto con numpy para no romper la reproducción |

**Requiere:** haber ejecutado el **Cuaderno A** (usa `outputs/embeddings_flickr8k.npz`).


In [ ]:
# 0. Setup
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import open_clip
from datasets import load_dataset

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_IMAGES = 200
MODEL_NAME = "ViT-B-32"
PRETRAINED = "laion2b_s34b_b79k"

os.makedirs("evidencias/zeroshot", exist_ok=True)
os.makedirs("evidencias/metricas", exist_ok=True)
os.makedirs("evidencias/rankings", exist_ok=True)
print("DEVICE:", DEVICE)

In [ ]:
# 1. Recargar datos y embeddings del Cuaderno A
dataset = load_dataset("jxie/flickr8k")
test_raw = dataset["test"][:N_IMAGES]
caption_cols = sorted([k for k in test_raw.keys() if k.startswith("caption_")])
meta = pd.DataFrame({
    "image_id": [f"test_{i}" for i in range(len(test_raw["image"]))],
    "image": test_raw["image"],
    "caption": test_raw[caption_cols[0]],
})

bundle = np.load("outputs/embeddings_flickr8k.npz", allow_pickle=True)
image_features = bundle["image_features"]
print("embeddings cargados:", image_features.shape,
      "| modelo:", bundle["model_name"], bundle["pretrained"])

#### 2. Etiquetas operativas para zero-shot

El bootstrap de **C9/C10** traía una columna `label` pequeña y curada *solo para demostración*.
Flickr8k no tiene clases, así que construyo etiquetas operativas por palabra clave en la caption
(adaptación declarada). Las imágenes sin etiqueta clara se excluyen del experimento zero-shot.


In [ ]:
# 2. Etiquetas por palabra clave (adaptacion propia del `label` del bootstrap)
KEYWORDS = {
    "dog":      ["dog", "dogs", "puppy"],
    "bicycle":  ["bike", "bicycle", "biker", "cyclist"],
    "water":    ["water", "pool", "lake", "ocean", "river", "swimming"],
    "snow":     ["snow", "snowy", "ski", "snowboard"],
    "football": ["football", "soccer", "ball"],
}

def derive_label(caption):
    text = caption.lower()
    matches = [lab for lab, kws in KEYWORDS.items() if any(k in text.split() or k in text for k in kws)]
    return matches[0] if len(matches) == 1 else None   # solo etiquetas no ambiguas

meta["label"] = meta["caption"].apply(derive_label)
zs = meta.dropna(subset=["label"]).reset_index()        # 'index' = posicion original (para indexar embeddings)
print(zs["label"].value_counts())
zs_image_features = image_features[zs["index"].values]

#### 3. Zero-shot: prompt único vs prompt ensemble

Adaptado de **C10** §4 y de la sección 3.1.4 del paper. La idea de CLIP: el clasificador se construye
**con lenguaje natural** — un embedding de texto por clase — sin reentrenar nada.

Detalle técnico defendible: tras **promediar** los embeddings de las plantillas hay que **re-normalizar**,
porque el promedio de vectores unitarios no es unitario.


In [ ]:
# 3. CELDA CLAVE: clasificador zero-shot construido con prompts
model, _, preprocess = open_clip.create_model_and_transforms(
    MODEL_NAME, pretrained=PRETRAINED, device=DEVICE)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model.eval()

TEMPLATES_SINGLE = ["a photo of a {}"]
TEMPLATES_ENSEMBLE = [
    "a photo of a {}",
    "a photo of {}",
    "a photo of people with a {}",     # plantilla propia
    "a low resolution photo of {}",    # plantilla propia
    "a close-up photo of {}",          # plantilla propia
]

LABELS = sorted(zs["label"].unique())

@torch.no_grad()
def class_embeddings(labels, templates):
    embs = []
    for label in labels:
        texts = [t.format(label) for t in templates]
        f = model.encode_text(tokenizer(texts).to(DEVICE))
        f = f / f.norm(dim=-1, keepdim=True)     # normalizar cada plantilla
        emb = f.mean(dim=0)                       # promedio del ensemble
        emb = emb / emb.norm()                    # RE-normalizar (clave)
        embs.append(emb.cpu().numpy())
    return np.stack(embs)

def zeroshot_eval(img_feats, class_embs, true_labels, labels):
    logits = img_feats @ class_embs.T
    preds = [labels[j] for j in logits.argmax(axis=1)]
    acc = float(np.mean([p == t for p, t in zip(preds, true_labels)]))
    return acc, preds

true_labels = zs["label"].tolist()
resultados = []
for nombre, templates in [("prompt unico", TEMPLATES_SINGLE), ("ensemble", TEMPLATES_ENSEMBLE)]:
    ce = class_embeddings(LABELS, templates)
    acc, preds = zeroshot_eval(zs_image_features, ce, true_labels, LABELS)
    resultados.append({"config": nombre, "n_templates": len(templates), "accuracy": round(acc, 4)})
    zs[f"pred_{nombre.replace(' ', '_')}"] = preds

tabla_zs = pd.DataFrame(resultados)
tabla_zs.to_csv("evidencias/zeroshot/zeroshot_prompt_summary.csv", index=False)
tabla_zs

In [ ]:
# Matriz de confusion (config ensemble) + separacion aciertos/errores
conf = pd.crosstab(zs["label"], zs["pred_ensemble"], rownames=["real"], colnames=["prediccion"])
conf.to_csv("evidencias/zeroshot/matriz_confusion.csv")

aciertos = zs[zs["label"] == zs["pred_ensemble"]]
errores = zs[zs["label"] != zs["pred_ensemble"]]
print(f"aciertos: {len(aciertos)} | errores: {len(errores)}")
errores[["image_id", "caption", "label", "pred_ensemble"]].to_csv(
    "evidencias/zeroshot/zeroshot_errores.csv", index=False)
conf

In [ ]:
# Inspeccionar un error de zero-shot (analisis de error cualitativo)
if len(errores) > 0:
    row = errores.iloc[0]
    plt.figure(figsize=(4, 4))
    plt.imshow(row["image"]); plt.axis("off")
    plt.title(f'real: {row["label"]}  |  pred: {row["pred_ensemble"]}\n{row["caption"][:60]}', fontsize=9)
    plt.savefig("evidencias/zeroshot/error_ejemplo.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Sin errores en este subconjunto (probable con clases faciles y pocas imagenes).")

#### 4. Comparación de checkpoints

Adaptado de **C10** §5. Mismo dataset, mismas consultas, **solo cambian los pesos** → la diferencia
es atribuible al preentrenamiento (LAION-2B vs WIT-400M de OpenAI). Conecta con la parte de *escala*
de mi tesis.


In [ ]:
# 4. Comparar checkpoints con las mismas imagenes y captions
from PIL import Image as PILImage

CONFIGS = [("ViT-B-32", "laion2b_s34b_b79k"), ("ViT-B-32", "openai")]
KS = (1, 5, 10)

def recall_at_k(sim_matrix, k):
    ranks = (-sim_matrix).argsort(axis=1)
    return float(np.mean([(ranks[i, :k] == i).any() for i in range(sim_matrix.shape[0])]))

@torch.no_grad()
def encode_with(model_name, pretrained, images, texts, batch_size=32):
    m, _, prep = open_clip.create_model_and_transforms(model_name, pretrained=pretrained, device=DEVICE)
    tok = open_clip.get_tokenizer(model_name)
    m.eval()
    img_f, txt_f = [], []
    for i in range(0, len(images), batch_size):
        b = torch.stack([prep(im.convert("RGB")) for im in images[i:i+batch_size]]).to(DEVICE)
        f = m.encode_image(b); img_f.append((f / f.norm(dim=-1, keepdim=True)).cpu())
    for i in range(0, len(texts), batch_size):
        f = m.encode_text(tok(texts[i:i+batch_size]).to(DEVICE))
        txt_f.append((f / f.norm(dim=-1, keepdim=True)).cpu())
    return torch.cat(img_f).numpy(), torch.cat(txt_f).numpy()

rows = []
for model_name, pretrained in CONFIGS:
    img_f, txt_f = encode_with(model_name, pretrained, meta["image"].tolist(), meta["caption"].tolist())
    s = img_f @ txt_f.T
    rows.append({"model": model_name, "pretrained": pretrained,
                 **{f"i2t_R@{k}": round(recall_at_k(s, k), 4) for k in KS},
                 **{f"t2i_R@{k}": round(recall_at_k(s.T, k), 4) for k in KS}})

cmp_df = pd.DataFrame(rows)
cmp_df.to_csv("evidencias/metricas/checkpoint_comparison.csv", index=False)
cmp_df

#### 5. Búsqueda semántica (FAISS con fallback exacto)

Adaptado de **C10** §6. Con embeddings normalizados, el índice `IndexFlatIP` (producto interno) equivale
a búsqueda por coseno. Si FAISS no está instalado, uso búsqueda exacta con numpy — mismo resultado en
este tamaño, lo que importa defender es **por qué el dual encoder permite indexar**: las imágenes se
codifican una sola vez; cada consulta nueva cuesta un solo `encode_text`.


In [ ]:
# 5. Indice semantico y consultas
try:
    import faiss
    index = faiss.IndexFlatIP(image_features.shape[1])
    index.add(image_features.astype("float32"))
    def search(query_feats, k=5):
        scores, idx = index.search(query_feats.astype("float32"), k)
        return idx[0], scores[0]
    print("usando FAISS IndexFlatIP")
except ImportError:
    def search(query_feats, k=5):
        scores = (query_feats @ image_features.T)[0]
        idx = np.argsort(-scores)[:k]
        return idx, scores[idx]
    print("FAISS no instalado: fallback exacto con numpy")

@torch.no_grad()
def encode_query(text):
    f = model.encode_text(tokenizer([text]).to(DEVICE))
    return (f / f.norm(dim=-1, keepdim=True)).cpu().numpy()

QUERIES = ["children playing in the snow", "a man riding a bicycle", "two dogs in the water"]
all_rows = []
for q in QUERIES:
    idx, scores = search(encode_query(q), k=5)
    for r, (j, s) in enumerate(zip(idx, scores), start=1):
        all_rows.append({"query": q, "rank": r, "image_id": meta.iloc[int(j)]["image_id"],
                         "caption": meta.iloc[int(j)]["caption"], "score": round(float(s), 4)})
faiss_df = pd.DataFrame(all_rows)
faiss_df.to_csv("evidencias/rankings/busqueda_semantica.csv", index=False)
faiss_df.head(10)

In [ ]:
# Visualizar la primera consulta
q = QUERIES[0]
idx, scores = search(encode_query(q), k=5)
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, j, s in zip(axes, idx, scores):
    ax.imshow(meta.iloc[int(j)]["image"]); ax.axis("off"); ax.set_title(f"{s:.3f}", fontsize=9)
fig.suptitle(f'Busqueda semantica: "{q}"')
plt.savefig("evidencias/rankings/busqueda_semantica_q1.png", dpi=150, bbox_inches="tight")
plt.show()

#### 6. Cierre

- **Zero-shot** verificado: el clasificador se construyó solo con prompts; el **ensemble** (paper §3.1.4: ~+3.5% en ImageNet con 80 prompts) se mide aquí en `zeroshot_prompt_summary.csv`.
- **Checkpoints**: la tabla aísla el efecto de los datos de preentrenamiento — argumento de escala de mi tesis.
- **Limitación:** etiquetas operativas por keyword son ruidosas (una caption no menciona todo lo que hay en la imagen) y las clases son pocas y fáciles; con un bootstrap pequeño las diferencias entre prompts pueden ser ruido.
- **Mejora:** más imágenes y seeds; clases más finas; ensembles con plantillas específicas por dominio.
